In [1]:
import codecs
import json
import os
from typing import Dict
import warnings

In [2]:
import numpy as np

In [3]:
data_dir = os.path.join('..', 'data', 'predicted', 'for_submission', 'with_metrics')
print(f'os.path.isdir({data_dir}) = {os.path.isdir(data_dir)}')

os.path.isdir(../data/predicted/for_submission/with_metrics) = True


In [4]:
names_of_submissions = sorted(list(filter(lambda it: it.endswith('.jsonl'), os.listdir(data_dir))))
print(f'There are {len(names_of_submissions)} submissions with estimated RL_F. They are:\n')
for it in names_of_submissions: print(it)

There are 7 submissions with estimated RL_F. They are:

submit_gemini-3-pro-preview-high_new_prompt.jsonl
submit_glm_46_gemini_prompt.jsonl
submit_haiku45_taskB.jsonl
submit_llama3_3_70b_gemini_prompt.jsonl
submit_meno_v17_ckp420_taskB.jsonl
submit_qwen235b_gemini_prompt_no_empty.jsonl
submit_qwen2_5_32b_taskB.jsonl


In [5]:
def load_metrics(fname: str) -> Dict[str, float]:
    with codecs.open(fname, mode='r', encoding='utf-8') as fp:
        samples = list(
            map(
                lambda it3: json.loads(it3),
                filter(
                    lambda it2: len(it2) > 0,
                    map(lambda it1: it1.strip(), fp.readlines())
                )
            )
        )
    scores_ = []
    for idx, val in enumerate(samples):
        err_msg = f'The file "{fname}": sample {idx} is wrong!\n{json.dumps(val, ensure_ascii=False, indent=4)}'
        ok = True
        rl_f = 0.0
        rl_f_idk = 0.0
        if 'metrics' not in val:
            raise ValueError(err_msg)
        if ('RL_F' not in val['metrics']) and ('RL_F_idk' not in val['metrics']):
            raise ValueError(err_msg)
        if 'RL_F' in val['metrics']:
            if len(val['metrics']['RL_F']) != 1:
                raise ValueError(err_msg)
            if isinstance(val['metrics']['RL_F'][0], float):
                rl_f = val['metrics']['RL_F'][0]
            else:
                ok = False
        if 'RL_F_idk' in val['metrics']:
            if len(val['metrics']['RL_F_idk']) != 1:
                raise ValueError(err_msg)
            if isinstance(val['metrics']['RL_F_idk'][0], float):
                rl_f_idk = val['metrics']['RL_F_idk'][0]
            else:
                ok = False
        if not ok:
            warnings.warn(err_msg)
            scores_.append((val['task_id'], max(rl_f, rl_f_idk)))
        else:
            if max(rl_f, rl_f_idk) < 1e-9:
                warnings.warn(err_msg)
                scores_.append((val['task_id'], 0.0))
            elif rl_f_idk > 1e-9:
                scores_.append((val['task_id'], rl_f_idk))
            else:
                scores_.append((val['task_id'], rl_f))
    return dict(scores_)

In [6]:
submission_metrics = [load_metrics(os.path.join(data_dir, it)) for it in names_of_submissions]

/tmp/ipykernel_45833/1305121676.py:41: UserWarning: The file "../data/predicted/for_submission/with_metrics/submit_gemini-3-pro-preview-high_new_prompt.jsonl": sample 2 is wrong!
{
    "conversation_id": "72013190593248ab20cca034f21ce38a",
    "task_id": "72013190593248ab20cca034f21ce38a<::>4",
    "task_type": "rag",
    "turn": 4,
    "dataset": "MT-RAG 2.0 Authors (Internal)",
    "contexts": [],
    "input": [
        {
            "speaker": "user",
            "text": "What are the different types of dialog nodes?",
            "metadata": {
                "author_type": "human",
                "author_id": "8c34bdda-13ea-46d5-8480-de474a5d3b4c",
                "created_at": 1723673431
            }
        },
        {
            "speaker": "agent",
            "text": "There are various types of dialog nodes. These include the Welcome node, which greets users when interacting with the assistant, and the Anything else node, which provides responses for unrecognized user inpu

In [7]:
all_task_ids = set(submission_metrics[0].keys())
for it in submission_metrics[1:]:
    all_task_ids |= set(it.keys())
all_task_ids = sorted(list(all_task_ids))
print(f'There are {len(all_task_ids)} tasks for submission.')

There are 507 tasks for submission.


In [8]:
total_results = dict()
for task_id in all_task_ids:
    selected_submission_indices = list(filter(lambda idx: task_id in submission_metrics[idx], range(len(names_of_submissions))))
    scores = [submission_metrics[model_idx][task_id] for model_idx in selected_submission_indices]
    total_results[task_id] = names_of_submissions[selected_submission_indices[np.argmax(scores)]]
    del scores, selected_submission_indices

In [9]:
frequencies_of_submissions = dict()
for task_id in total_results:
    if total_results[task_id] in frequencies_of_submissions:
        frequencies_of_submissions[total_results[task_id]] += 1
    else:
        frequencies_of_submissions[total_results[task_id]] = 1

In [10]:
max_text_width = max([len(it) for it in names_of_submissions])

In [11]:
for it in names_of_submissions:
    print('{0:>{1}}: {2:>3}'.format(it, max_text_width, frequencies_of_submissions[it]))

submit_gemini-3-pro-preview-high_new_prompt.jsonl: 405
                submit_glm_46_gemini_prompt.jsonl:  33
                       submit_haiku45_taskB.jsonl:  28
          submit_llama3_3_70b_gemini_prompt.jsonl:   5
               submit_meno_v17_ckp420_taskB.jsonl:  13
     submit_qwen235b_gemini_prompt_no_empty.jsonl:   1
                   submit_qwen2_5_32b_taskB.jsonl:  22
